# Projeto de Parceria | Semantix — SVM x XGBoost

## Classificação de tumores de mama

Objetivo: comparar SVM e XGBoost no Breast Cancer Wisconsin Diagnostic Dataset. Este é um estudo acadêmico de Machine Learning e não uma ferramenta de diagnóstico médico.


## 1. Coleta de dados
O conjunto Breast Cancer Wisconsin (Diagnostic) é público e está disponível pela UCI Machine Learning Repository e pelo `scikit-learn`. Nesta versão, usamos `load_breast_cancer`, que fornece 569 amostras, 30 características numéricas e a variável alvo (0 = maligno; 1 = benigno).


In [1]:
import pandas as pd
from sklearn.datasets import load_breast_cancer
data = load_breast_cancer(as_frame=True)
df = data.frame.copy()
print(df.shape)
print('Valores ausentes:', df.isna().sum().sum())
df.head()


(569, 31)
Valores ausentes: 0


,mean radius,mean texture,mean perimeter,mean area,mean smoothness,mean compactness,mean concavity,mean concave points,mean symmetry,mean fractal dimension,radius error,texture error,perimeter error,area error,smoothness error,compactness error,concavity error,concave points error,symmetry error,fractal dimension error,worst radius,worst texture,worst perimeter,worst area,worst smoothness,worst compactness,worst concavity,worst concave points,worst symmetry,worst fractal dimension,target
0,17.99,10.38,122.80,1001.0,0.11840,0.27760,0.3001,0.14710,0.2419,0.07871,1.0950,0.9053,8.589,153.40,0.006399,0.04904,0.05373,0.01587,0.03003,0.006193,25.38,17.33,184.60,2019.0,0.1622,0.6656,0.7119,0.2654,0.4601,0.11890,0
1,20.57,17.77,132.90,1326.0,0.08474,0.07864,0.0869,0.07017,0.1812,0.05667,0.5435,0.7339,3.398,74.08,0.005225,0.01308,0.01860,0.01340,0.01389,0.003532,24.99,23.41,158.80,1956.0,0.1238,0.1866,0.2416,0.1860,0.2750,0.08902,0
2,19.69,21.25,130.00,1203.0,0.10960,0.15990,0.1974,0.12790,0.2069,0.05999,0.7456,0.7869,4.585,94.03,0.006150,0.04006,0.03832,0.02058,0.02250,0.004571,23.57,25.53,152.50,1709.0,0.1444,0.4245,0.4504,0.2430,0.3613,0.08758,0
3,11.42,20.38,77.58,386.1,0.14250,0.28390,0.2414,0.10520,0.2597,0.09744,0.4956,1.1560,3.445,27.23,0.009110,0.07458,0.05661,0.01867,0.05963,0.009208,14.91,26.50,98.87,567.7,0.2098,0.8663,0.6869,0.2575,0.6638,0.17300,0
4,20.29,14.34,135.10,1297.0,0.10030,0.13280,0.1980,0.10430,0.1809,0.05883,0.7572,0.7813,5.438,94.44,0.011490,0.02461,0.05688,0.01885,0.01756,0.005115,22.54,16.67,152.20,1575.0,0.1374,0.2050,0.4000,0.1625,0.2364,0.07678,0


## 2. Análise exploratória
Verificamos dimensões, valores ausentes, estatísticas descritivas e distribuição das classes.


In [2]:
print(df['target'].value_counts())
df.describe().T


target
1    357
0    212
Name: count, dtype: int64


,count,mean,std,min,25%,50%,75%,max
mean radius,569.0,14.127292,3.524049,6.981000,11.700000,13.370000,15.780000,28.11000
mean texture,569.0,19.289649,4.301036,9.710000,16.170000,18.840000,21.800000,39.28000
mean perimeter,569.0,91.969033,24.298981,43.790000,75.170000,86.240000,104.100000,188.50000
mean area,569.0,654.889104,351.914129,143.500000,420.300000,551.100000,782.700000,2501.00000
mean smoothness,569.0,0.096360,0.014064,0.052630,0.086370,0.095870,0.105300,0.16340
mean compactness,569.0,0.104341,0.052813,0.019380,0.064920,0.092630,0.130400,0.34540
mean concavity,569.0,0.088799,0.079720,0.000000,0.029560,0.061540,0.130700,0.42680
mean concave points,569.0,0.048919,0.038803,0.000000,0.020310,0.033500,0.074000,0.20120
mean symmetry,569.0,0.181162,0.027414,0.106000,0.161900,0.179200,0.195700,0.30400
mean fractal dimension,569.0,0.062798,0.007060,0.049960,0.057700,0.061540,0.066120,0.09744


## 3. Preparação e divisão dos dados
Usamos 80% para treino e 20% para teste, com estratificação. No SVM, a padronização é feita dentro de um `Pipeline`, evitando vazamento de dados durante a validação cruzada.


In [3]:
from sklearn.model_selection import train_test_split, StratifiedKFold
X = df.drop(columns='target'); y = df['target']
X_train, X_test, y_train, y_test = train_test_split(X,y,test_size=0.20,random_state=42,stratify=y)
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)


## 4. Modelagem — SVM
O SVM foi combinado com `StandardScaler` e otimizado por `GridSearchCV`, usando ROC-AUC como métrica de seleção.


In [4]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.model_selection import GridSearchCV
svm_pipe = Pipeline([('scaler', StandardScaler()),('svc', SVC(probability=True, random_state=42))])
svm_grid = {'svc__C':[0.1,1,10,100], 'svc__gamma':['scale',0.001,0.01,0.1], 'svc__kernel':['rbf']}
svm_search = GridSearchCV(svm_pipe, svm_grid, scoring='roc_auc', cv=cv, n_jobs=-1)
svm_search.fit(X_train,y_train)
svm_search.best_params_


{'svc__C': 10, 'svc__gamma': 0.01, 'svc__kernel': 'rbf'}

## 5. Modelagem — XGBoost
O XGBoost foi otimizado separadamente, avaliando profundidade, taxa de aprendizado, número de árvores e amostragem.


In [5]:
from xgboost import XGBClassifier
xgb = XGBClassifier(objective='binary:logistic',eval_metric='logloss',random_state=42,n_jobs=1,tree_method='hist')
xgb_grid = {'n_estimators':[100,200], 'max_depth':[2,3,4], 'learning_rate':[0.03,0.05,0.1], 'subsample':[0.8,1.0], 'colsample_bytree':[0.8,1.0]}
xgb_search = GridSearchCV(xgb, xgb_grid, scoring='roc_auc', cv=cv, n_jobs=-1)
xgb_search.fit(X_train,y_train)
xgb_search.best_params_


{'colsample_bytree': 0.8,
 'learning_rate': 0.05,
 'max_depth': 2,
 'n_estimators': 200,
 'subsample': 0.8}

## 6. Avaliação e visualização
Comparamos Accuracy, Precision, Recall, F1 e ROC-AUC no conjunto de teste, além de matrizes de confusão e curva ROC.


In [6]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
resultados=[]
for nome,modelo in {'SVM':svm_search.best_estimator_,'XGBoost':xgb_search.best_estimator_}.items():
    pred=modelo.predict(X_test); prob=modelo.predict_proba(X_test)[:,1]
    resultados.append([nome,accuracy_score(y_test,pred),precision_score(y_test,pred),recall_score(y_test,pred),f1_score(y_test,pred),roc_auc_score(y_test,prob)])
pd.DataFrame(resultados,columns=['Modelo','Accuracy','Precision','Recall','F1','ROC_AUC'])


,Modelo,Accuracy,Precision,Recall,F1,ROC_AUC
0,SVM,0.982456,0.986111,0.986111,0.986111,0.997685
1,XGBoost,0.956140,0.946667,0.986111,0.965986,0.993717


## 7. Resultados obtidos

| Modelo | Accuracy | Precision | Recall | F1 | ROC-AUC |
|---|---:|---:|---:|---:|---:|
| SVM | 0.9825 | 0.9861 | 0.9861 | 0.9861 | 0.9977 |
| XGBoost | 0.9561 | 0.9467 | 0.9861 | 0.9660 | 0.9937 |

Melhores parâmetros SVM: `{'svc__C': 10, 'svc__gamma': 0.01, 'svc__kernel': 'rbf'}`.

Melhores parâmetros XGBoost: `{'colsample_bytree': 0.8, 'learning_rate': 0.05, 'max_depth': 2, 'n_estimators': 200, 'subsample': 0.8}`.


## 8. Conclusões
Neste experimento, o SVM apresentou o melhor desempenho geral no conjunto de teste, especialmente em Accuracy, F1 e ROC-AUC. O XGBoost também apresentou desempenho elevado e Recall equivalente. Os resultados reforçam que a escolha do algoritmo deve ser orientada por validação experimental, e não pela suposição de que um método mais complexo será necessariamente superior.

Como limitação, o conjunto é relativamente pequeno. Uma continuação natural seria repetir a avaliação com validação cruzada aninhada, testar calibração e avaliar outros conjuntos de dados.

**Observação:** este trabalho tem finalidade acadêmica e não deve ser utilizado para diagnóstico médico.
